# 00 — 数据状态观测

**每次分析前先跑**，确认各类数据就绪状态。

| Section | 内容 |
|---------|------|
| 1 | 状态汇总表（颜色编码）|
| 2 | K 线覆盖热力图 |
| 3 | 数据时效横条图 |
| 4 | 板块成分完整性 |
| 5 | 建议操作输出 |

In [1]:
# ── 初始化 ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = PROJECT_ROOT / 'data'

import pandas as pd
import numpy as np
from datetime import date, timedelta

TODAY = date.today()
print(f'Project: {PROJECT_ROOT}')
print(f'Data:    {DATA_ROOT}')
print(f'Today:   {TODAY}')

Project: /data00/home/guohuanwei.cztj/git_files/trade
Data:    /data00/home/guohuanwei.cztj/git_files/trade/data
Today:   2026-03-09


In [2]:
# ── 工具函数 ──────────────────────────────────────────────────────────────────
import sqlite3

def _parquet_files(subdir: str):
    d = DATA_ROOT / subdir
    if not d.exists():
        return []
    return list(d.glob('*.parquet'))

def _parquet_stats(files):
    """Returns (count, oldest_date_str, newest_date_str, days_since_newest)"""
    if not files:
        return 0, None, None, None
    min_date, max_date = None, None
    for f in files:
        try:
            df = pd.read_parquet(f)
            date_col = next((c for c in ['date', 'trade_date', 'ann_date'] if c in df.columns), None)
            if date_col is None:
                continue
            dates = pd.to_datetime(df[date_col], errors='coerce').dropna()
            if dates.empty:
                continue
            mn = dates.min().date()
            mx = dates.max().date()
            if min_date is None or mn < min_date:
                min_date = mn
            if max_date is None or mx > max_date:
                max_date = mx
        except Exception:
            pass
    if max_date is None:
        return len(files), None, None, None
    days = (TODAY - max_date).days
    return len(files), str(min_date) if min_date else None, str(max_date), days

def _kline_stats():
    """Count stocks with kline parquets."""
    kline_dir = DATA_ROOT / 'kline'
    if not kline_dir.exists():
        return 0, None, None, None
    files = [f for f in kline_dir.glob('*.parquet') if not f.name.startswith('.')]
    return _parquet_stats(files)

def _sector_index_stats():
    idx_dir = DATA_ROOT / 'index'
    if not idx_dir.exists():
        return 0, None, None, None
    files = list(idx_dir.glob('sector_*.parquet'))
    return len(files), None, None, None  # date scan skipped for speed

def _db_instrument_stats():
    db_path = DATA_ROOT / '.metadata' / 'trade.db'
    if not db_path.exists():
        return 0, 0
    with sqlite3.connect(str(db_path)) as conn:
        total = conn.execute('SELECT COUNT(*) FROM instruments').fetchone()[0]
        with_industry = conn.execute('SELECT COUNT(*) FROM instruments WHERE industry > 0').fetchone()[0]
    return total, with_industry

def _sentiment_stats():
    sent_dir = DATA_ROOT / 'sentiment'
    if not sent_dir.exists():
        return 0, None, None, None
    files = list(sent_dir.glob('*.parquet')) + list(sent_dir.glob('**/*.parquet'))
    return _parquet_stats(files)

print('工具函数已加载')

工具函数已加载


## Section 1 — 状态汇总表

In [4]:
from IPython.display import display

# 采集各类数据状态
kline_n, kline_old, kline_new, kline_days = _kline_stats()
fund_files = _parquet_files('fundamental')
fund_n, fund_old, fund_new, fund_days = _parquet_stats(fund_files)
ff_files = _parquet_files('fund_flow')
ff_n, ff_old, ff_new, ff_days = _parquet_stats(ff_files)
nb_files = _parquet_files('northbound')
nb_n, nb_old, nb_new, nb_days = _parquet_stats(nb_files)
idx_dir = DATA_ROOT / 'index'
broad_files = [f for f in idx_dir.glob('*.parquet') if not f.name.startswith('sector_')] if idx_dir.exists() else []
broad_n, broad_old, broad_new, broad_days = _parquet_stats(broad_files)
sec_n, _, _, _ = _sector_index_stats()
macro_files = list((DATA_ROOT / 'macro').glob('*.parquet')) if (DATA_ROOT / 'macro').exists() else []
macro_n, macro_old, macro_new, macro_days = _parquet_stats(macro_files)
sent_n, sent_old, sent_new, sent_days = _sentiment_stats()
total_instr, instr_with_ind = _db_instrument_stats()

def _status(n, days):
    if n == 0:
        return '🔴 无'
    if days is None:
        return '🟡 有'
    if days <= 3:
        return '🟢'
    if days <= 14:
        return '🟡'
    return '🟠'

rows = [
    {'数据类型': 'K 线',    '状态': _status(kline_n, kline_days),  '数量': f'{kline_n} 只',  '最旧': kline_old or '—', '最新': kline_new or '—', '距今(天)': kline_days if kline_days is not None else '—', '同步命令': '— (已就绪)' if kline_n > 0 else './trade data kline sync'},
    {'数据类型': '财务数据', '状态': _status(fund_n, fund_days),   '数量': f'{fund_n} 只',   '最旧': fund_old or '—',  '最新': fund_new or '—',  '距今(天)': fund_days if fund_days is not None else '—',  '同步命令': '— (已就绪)' if fund_n > 0 else './trade data fundamental sync'},
    {'数据类型': '资金流向', '状态': _status(ff_n, ff_days),       '数量': f'{ff_n} 只',     '最旧': ff_old or '—',    '最新': ff_new or '—',    '距今(天)': ff_days if ff_days is not None else '—',    '同步命令': '— (已就绪)' if ff_n > 0 else './trade data fund-flow sync'},
    {'数据类型': '北向资金', '状态': _status(nb_n, nb_days),       '数量': f'{nb_n} 文件',   '最旧': nb_old or '—',    '最新': nb_new or '—',    '距今(天)': nb_days if nb_days is not None else '—',    '同步命令': '— (已就绪)' if nb_n > 0 else './trade data northbound sync'},
    {'数据类型': '宽基指数', '状态': _status(broad_n, broad_days), '数量': f'{broad_n} 个',  '最旧': broad_old or '—', '最新': broad_new or '—', '距今(天)': broad_days if broad_days is not None else '—', '同步命令': '— (已就绪)' if broad_n > 0 else './trade data index sync'},
    {'数据类型': '板块指数', '状态': _status(sec_n, None),         '数量': f'{sec_n}/31 个',  '最旧': '—',              '最新': '—',              '距今(天)': '—',                                             '同步命令': '— (已就绪)' if sec_n >= 31 else './trade data index sync-sector'},
    {'数据类型': '宏观数据', '状态': _status(macro_n, macro_days), '数量': f'{macro_n} 文件','最旧': macro_old or '—', '最新': macro_new or '—', '距今(天)': macro_days if macro_days is not None else '—', '同步命令': '— (已就绪)' if macro_n > 0 else './trade data macro sync'},
    {'数据类型': '情绪数据', '状态': _status(sent_n, sent_days),   '数量': f'{sent_n} 文件', '最旧': sent_old or '—',  '最新': sent_new or '—',  '距今(天)': sent_days if sent_days is not None else '—',  '同步命令': '— (已就绪)' if sent_n > 0 else './trade data sentiment'},
]

summary = pd.DataFrame(rows)

def _color_status(v):
    if '🔴' in str(v):
        return 'background-color: #ffcccc'
    if '🟠' in str(v):
        return 'background-color: #ffe0b2'
    if '🟡' in str(v):
        return 'background-color: #fff9c4'
    if '🟢' in str(v):
        return 'background-color: #c8e6c9'
    return ''

styled = summary.style.applymap(_color_status, subset=['状态'])
display(styled)

,数据类型,状态,数量,最旧,最新,距今(天),同步命令
0,K 线,🔴 无,0 只,—,—,—,./trade data kline sync
1,财务数据,🔴 无,0 只,—,—,—,./trade data fundamental sync
2,资金流向,🔴 无,0 只,—,—,—,./trade data fund-flow sync
3,北向资金,🔴 无,0 文件,—,—,—,./trade data northbound sync
4,宽基指数,🔴 无,0 个,—,—,—,./trade data index sync
5,板块指数,🔴 无,0/31 个,—,—,—,./trade data index sync-sector
6,宏观数据,🔴 无,0 文件,—,—,—,./trade data macro sync
7,情绪数据,🟢,10 文件,2026-03-02,2026-03-06,3,— (已就绪)


## Section 2 — K 线覆盖热力图（月份 × 股票样本）

In [ ]:
import plotly.graph_objects as go

kline_dir = DATA_ROOT / 'kline'
if not kline_dir.exists() or kline_n == 0:
    print('⚠ 无 K 线数据，跳过热力图')
else:
    kline_files = sorted(kline_dir.glob('*.parquet'))[:50]  # 最多显示50只
    month_bar_counts = {}
    symbols = []
    for f in kline_files:
        sym = f.stem
        try:
            df = pd.read_parquet(f)
            if 'date' not in df.columns:
                continue
            df['month'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
            mc = df.groupby('month').size()
            month_bar_counts[sym] = mc
            symbols.append(sym)
        except Exception:
            pass

    if month_bar_counts:
        all_months = sorted(set().union(*[set(v.index) for v in month_bar_counts.values()]))
        all_months = all_months[-36:]  # 最近3年
        z = []
        for sym in symbols:
            row = [month_bar_counts[sym].get(m, 0) for m in all_months]
            z.append(row)

        fig = go.Figure(go.Heatmap(
            z=z,
            x=all_months,
            y=symbols,
            colorscale='Greens',
            colorbar=dict(title='Bar 数'),
            hoverongaps=False,
        ))
        fig.update_layout(
            title=f'K 线月度覆盖热力图（样本 {len(symbols)} 只）',
            xaxis_title='月份',
            yaxis_title='股票',
            height=max(300, len(symbols) * 12 + 100),
        )
        fig.show()
    else:
        print('无可读取的 K 线数据')

## Section 3 — 数据时效横条图

In [ ]:
import plotly.express as px

staleness = [
    {'类型': 'K 线',    '距今(天)': kline_days if kline_days is not None else -1},
    {'类型': '财务数据', '距今(天)': fund_days  if fund_days  is not None else -1},
    {'类型': '资金流向', '距今(天)': ff_days    if ff_days    is not None else -1},
    {'类型': '北向资金', '距今(天)': nb_days    if nb_days    is not None else -1},
    {'类型': '宽基指数', '距今(天)': broad_days if broad_days is not None else -1},
    {'类型': '宏观数据', '距今(天)': macro_days if macro_days is not None else -1},
    {'类型': '情绪数据', '距今(天)': sent_days  if sent_days  is not None else -1},
]
df_stale = pd.DataFrame(staleness)
df_stale = df_stale[df_stale['距今(天)'] >= 0].sort_values('距今(天)')

if df_stale.empty:
    print('⚠ 暂无任何有效数据，请先同步')
else:
    def _bar_color(days):
        if days <= 3:  return '#4CAF50'
        if days <= 14: return '#FFC107'
        return '#F44336'

    colors = [_bar_color(d) for d in df_stale['距今(天)']]
    fig = px.bar(
        df_stale, x='距今(天)', y='类型', orientation='h',
        title='各类数据距今天数（越小越新鲜）',
        color='距今(天)',
        color_continuous_scale=['#4CAF50', '#FFC107', '#F44336'],
    )
    fig.update_layout(showlegend=False)
    fig.show()

## Section 4 — 板块成分完整性

In [ ]:
db_path = DATA_ROOT / '.metadata' / 'trade.db'
if not db_path.exists():
    print('⚠ trade.db 不存在')
else:
    with sqlite3.connect(str(db_path)) as conn:
        df_instr = pd.read_sql('SELECT symbol, name, industry FROM instruments ORDER BY symbol', conn)

    total = len(df_instr)
    unknown = (df_instr['industry'] == 0).sum()
    pct_ok = (total - unknown) / total * 100 if total > 0 else 0

    print(f'总标的数: {total}')
    print(f'已映射板块: {total - unknown} ({pct_ok:.1f}%)')
    print(f'未映射 (industry=0): {unknown}')

    if unknown > 0:
        print('\n未映射样本（前20）:')
        display(df_instr[df_instr['industry'] == 0].head(20))
    else:
        print('\n✅ 所有标的均已映射到 SW 板块')

    # 板块分布饼图
    try:
        from trade_py.analysis.knowledge_graph import SW, SW_NAMES_ZH
        df_instr['industry_name'] = df_instr['industry'].apply(
            lambda x: SW_NAMES_ZH.get(SW(int(x)), f'unknown({x})') if x > 0 else '未映射'
        )
        dist = df_instr.groupby('industry_name').size().reset_index(name='count').sort_values('count', ascending=False)
        fig = px.bar(dist, x='industry_name', y='count', title='标的板块分布',
                     labels={'industry_name': '板块', 'count': '股票数'})
        fig.update_xaxes(tickangle=45)
        fig.show()
    except ImportError:
        print('（knowledge_graph 未加载，跳过分布图）')

## Section 5 — 建议操作

In [ ]:
suggestions = []

if kline_n == 0:
    suggestions.append(('K 线', './trade data kline sync --mode full --start 2020-01-01'))
elif kline_days is not None and kline_days > 3:
    suggestions.append(('K 线（过期）', './trade data kline sync'))

if fund_n == 0:
    suggestions.append(('财务数据', './trade data fundamental sync'))

if ff_n == 0:
    suggestions.append(('资金流向', './trade data fund-flow sync --start 2024-01-01'))

if nb_n == 0:
    suggestions.append(('北向资金', './trade data northbound sync --start 2023-01-01'))

if broad_n == 0:
    suggestions.append(('宽基指数', './trade data index sync'))

if sec_n < 31:
    suggestions.append(('板块指数', f'./trade data index sync-sector --start 2023-01-01  # 当前 {sec_n}/31 个'))

if macro_n == 0:
    suggestions.append(('宏观数据', './trade data macro sync'))

if total_instr > 0 and instr_with_ind < total_instr * 0.9:
    suggestions.append(('板块成分映射', './trade data index refresh-members'))

if suggestions:
    print('📋 建议执行以下同步命令：\n')
    for label, cmd in suggestions:
        print(f'  [{label}]')
        print(f'    {cmd}\n')
else:
    print('✅ 所有数据就绪，可直接开始分析！')